In [13]:
import pandas as pd
import os

df_sample = pd.read_csv("../../config/samplesTestFast.csv")
df_sample

,sample_name,sample_run,sample_path,image_path,image_param,cytaimage_param,cytaimage_path,slide_param,slide_value,align_param,align_value,probe_param,probe_value,area_param,area_value
0,tinytest,tiny_run,/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample...,/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample...,--image=,,,--unknown-slide=,visium-1,,,,,,
1,tinytest2,tiny_run,/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample...,/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample...,--image=,,,--unknown-slide=,visium-1,,,,,,


In [14]:
# --- implementation for single runs ---

# 1. Sort the DataFrame by 'sample_name' and 'sample_run'
df_sample2 = df_sample.sort_values(by=['sample_name', 'sample_run']).reset_index(drop=True)

# 2. Create a cumulative count within each 'sample_name' group: cumcount() starts at 0, so add 1 to start numbering from 1
df_sample2['sample_instance_num'] = df_sample2.groupby('sample_name').cumcount() + 1

# 3. Create the new unique sample name string
df_sample2['sample_name_unique'] = df_sample2['sample_name'] + '_' + df_sample2['sample_instance_num'].astype(str)

# generate a pandas dataframe. Here it is expected only one path per sample/run.
aggregated_single_alt = df_sample2.groupby(['sample_name','sample_run']).agg(
    sample_run=('sample_run', 'unique'),  # Get unique 'sample_run' per group
    sample_path_string=('sample_path', lambda s: sorted(s.map(os.path.dirname).unique())),
    # needed to check for the existance of the actual input files
    sample_path_list=('sample_path','unique'),
    # needed to avoid issues with the *.mro files during processing
    sample_name_unique=('sample_name_unique', 'first'),
    # add the param image. this can be either "image" or "darkimage". it is mandatory
    param_image=('image_param','first'),
    # add the image path
    sample_image=('image_path','first'),
    # add the param cytimage. this can be either "" or "--cytaimage=".
    param_cytaimage=('cytaimage_param','first'),
    # add the image path
    sample_cytaimage=('cytaimage_path','first'),
    # add the slide parameter. this can be either "--unknown-slide=" or "--slide=".
    param_slide=('slide_param','first'),
    # add the slide value.
    value_slide=('slide_value','first'),
    # add the alignament parameter. this can be either "" or "--loupe-alignment=".
    param_align=('align_param','first'),
    # add the slide value
    value_align=('align_value','first'),
    # add the probe parameter. this can be either "" or "--probe-set=".
    param_probe=('probe_param','first'),
    # add the probe path
    value_probe=('probe_value','first'),
    # add the area parameter. this can be either "" or "--probe-set=".
    param_area=('area_param','first'),
    # add the area value
    value_area=('area_value','first')
)

# 3. Convert the aggregated DataFrame to the desired dictionary format
SAMPLES_single = aggregated_single_alt.to_dict('index')


In [15]:
SAMPLES_single

{('tinytest', 'tiny_run'): {'sample_run': array(['tiny_run'], dtype=object),
  'sample_path_string': ['/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample_data/sample_spatial/spaceranger_tiny_inputs/fastqs'],
  'sample_path_list': array(['/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample_data/sample_spatial/spaceranger_tiny_inputs/fastqs/tinytest'],
        dtype=object),
  'sample_name_unique': 'tinytest_1',
  'param_image': '--image=',
  'sample_image': '/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample_data/sample_spatial/spaceranger_tiny_inputs/image/tinyimage.jpg',
  'param_cytaimage': ' ',
  'sample_cytaimage': ' ',
  'param_slide': '--unknown-slide=',
  'value_slide': 'visium-1',
  'param_align': ' ',
  'value_align': ' ',
  'param_probe': ' ',
  'value_probe': ' ',
  'param_area': ' ',
  'value_area': ' '},
 ('tinytest2', 'tiny_run'): {'sample_run': array(['tiny_run'], dtype=object),
  'sample_path_string': ['/idle/ric.cosr/ric.cosr/pedrini.edoardo/sample_data/sample_spatial/spaceranger_t